In [ ]:
"""
IS 456:2000 compliance layer — Concrete Mix Design Assistant
Concreate Club ML Inductions, IIT Indore

Civil-engineering rules, deliberately DECOUPLED from the ML strength
prediction, exactly as the problem statement mandates:

  * W/C ratio = Water / Cement ONLY. Fly ash and blast furnace slag are
    NOT binder for this calculation — they are excluded from the
    denominator, consistent with IS 456:2000 Table 5 as written.

  * IS 456 Table 5 (values required by the PS — implement exactly):

        Grade | Min strength (MPa) | Min cement (kg/m3) | Max W/C
        ------|--------------------|--------------------|--------
        M20   |        20          |        300         |  0.55
        M25   |        25          |        300         |  0.50
        M30   |        30          |        320         |  0.45
        M35   |        35          |        340         |  0.45
        M40   |        40          |        360         |  0.40

  * Key engineering case the app must catch: a mix can predict ADEQUATE
    strength (e.g. strength boosted by slag/fly ash) while still FAILING
    IS 456 compliance (too little plain cement, or W/C too high). The
    three verdicts — strength, cement content, W/C ratio — are therefore
    reported separately so the user never confuses a strength failure
    with a compliance failure.

Self-test:
    python src/compliance.py
"""

from dataclasses import dataclass, asdict, field

# IS 456:2000 Table 5 — problem-statement values, implemented exactly
IS_456_TABLE_5 = {
    "M20": {"min_strength": 20.0, "min_cement": 300.0, "max_wc": 0.55},
    "M25": {"min_strength": 25.0, "min_cement": 300.0, "max_wc": 0.50},
    "M30": {"min_strength": 30.0, "min_cement": 320.0, "max_wc": 0.45},
    "M35": {"min_strength": 35.0, "min_cement": 340.0, "max_wc": 0.45},
    "M40": {"min_strength": 40.0, "min_cement": 360.0, "max_wc": 0.40},
}
GRADES = list(IS_456_TABLE_5)


@dataclass
class ComplianceResult:
    """Separate verdicts so the UI can display strength vs compliance failures distinctly."""
    target_grade: str
    predicted_grade: str
    predicted_strength: float
    strength_pass: bool
    cement_pass: bool
    wc_pass: bool
    actual_wc: float
    min_cement_required: float
    max_wc_allowed: float
    overall_pass: bool = field(init=False)

    def __post_init__(self) -> None:
        self.overall_pass = self.strength_pass and self.cement_pass and self.wc_pass

    def to_dict(self) -> dict:
        return asdict(self)


def classify_grade(strength: float) -> str:
    """Map a predicted strength (MPa) to its IS 456 grade band."""
    if strength >= 40.0:
        return "M40+"
    if strength >= 35.0:
        return "M35"
    if strength >= 30.0:
        return "M30"
    if strength >= 25.0:
        return "M25"
    if strength >= 20.0:
        return "M20"
    return "Below Grade (<M20)"


def wc_ratio(water: float, cement: float) -> float:
    """Water-cement ratio: water / cement only. Guards division by zero."""
    if cement <= 0:
        return float("inf")
    return water / cement


def verify_is456(
    target_grade: str,
    predicted_strength: float,
    cement: float,
    water: float,
) -> ComplianceResult:
    """Check a mix against IS 456 Table 5 for the user's target grade."""
    if target_grade not in IS_456_TABLE_5:
        raise ValueError(f"Unknown grade {target_grade!r}. Valid grades: {GRADES}")
    limits = IS_456_TABLE_5[target_grade]
    ratio = wc_ratio(water, cement)

    return ComplianceResult(
        target_grade=target_grade,
        predicted_grade=classify_grade(predicted_strength),
        predicted_strength=round(predicted_strength, 2),
        strength_pass=predicted_strength >= limits["min_strength"],
        cement_pass=cement >= limits["min_cement"],
        wc_pass=ratio <= limits["max_wc"],
        actual_wc=round(ratio, 3),
        min_cement_required=limits["min_cement"],
        max_wc_allowed=limits["max_wc"],
    )


# ------------------------------------------------------------------ self-test
if __name__ == "__main__":
    # 1) Fully compliant M25 mix
    r = verify_is456("M25", 26.5, 310.0, 150.0)          # wc = 0.484 <= 0.50
    assert r.strength_pass and r.cement_pass and r.wc_pass and r.overall_pass

    # 2) Strong mix that FAILS W/C compliance — the decoupling case.
    #    Strength 27 MPa passes M25, but wc = 180/300 = 0.60 > 0.50 fails.
    r = verify_is456("M25", 27.0, 300.0, 180.0)
    assert r.strength_pass and not r.wc_pass and not r.overall_pass

    # 3) Strength shortfall with compliant cement and W/C
    r = verify_is456("M30", 28.0, 330.0, 145.0)          # wc = 0.439 <= 0.45
    assert not r.strength_pass and r.cement_pass and r.wc_pass

    # 4) Grade classification boundaries
    assert classify_grade(40.0) == "M40+"
    assert classify_grade(39.9) == "M35"
    assert classify_grade(25.0) == "M25"
    assert classify_grade(19.9) == "Below Grade (<M20)"

    # 5) Zero-cement guard (invalid input must not crash with ZeroDivisionError)
    assert wc_ratio(180.0, 0.0) == float("inf")

    print("All IS 456:2000 self-tests passed.")


All IS 456:2000 self-tests passed.


In [ ]:
"""
Model inference layer — Concrete Mix Design Assistant
Loads the Phase 1 artifacts and exposes a single validated predict() call.

Contract (per the problem statement):
  * The app accepts 7 mix ingredients only; age is ALWAYS injected as 28 days.
  * Inputs are validated (finite, non-negative) so bad UI input fails with a
    clear ValueError instead of a silent garbage prediction. The Streamlit
    app catches these and shows clean error messages.
"""

from __future__ import annotations

import math
from pathlib import Path

import joblib
import pandas as pd

DEFAULT_MODEL_PATH = Path("models/model.joblib")
FIXED_AGE = 28.0  # PS: all app predictions are at the fixed 28-day point

INGREDIENTS = [
    "cement", "slag", "fly_ash", "water", "superplasticizer",
    "coarse_aggregate", "fine_aggregate",
]


class ModelHandler:
    def __init__(self, model_path: str | Path = DEFAULT_MODEL_PATH) -> None:
        path = Path(model_path)
        if not path.exists():
            raise FileNotFoundError(
                f"{path} not found — run train_model.py first (Phase 1)."
            )
        payload = joblib.load(path)
        self.model = payload["model"]
        self.features = payload["features"]  # exact training column-order contract
        self.model_type = payload.get("model_type", "unknown")

    def predict(self, mix: dict) -> float:
        """Predict 28-day compressive strength (MPa) for a 7-ingredient mix dict."""
        row = {}
        for name in self.features:
            if name == "age":
                row["age"] = FIXED_AGE
                continue
            if name not in mix:
                raise ValueError(f"Missing ingredient: {name!r}")
            raw = mix[name]
            try:
                value = float(raw)
            except (TypeError, ValueError):
                raise ValueError(f"Invalid value for {name!r}: {raw!r}")
            if not math.isfinite(value):
                raise ValueError(f"Invalid value for {name!r}: {raw!r}")
            if value < 0:
                raise ValueError(f"{name!r} must be non-negative, got {value}")
            row[name] = value
        X = pd.DataFrame([row])[self.features]  # enforce training column order
        return float(self.model.predict(X)[0])


if __name__ == "__main__":
    handler = ModelHandler()
    demo = {
        "cement": 350.0, "slag": 100.0, "fly_ash": 0.0, "water": 190.0,
        "superplasticizer": 8.0, "coarse_aggregate": 1000.0, "fine_aggregate": 752.0,
    }
    print(f"Model: {handler.model_type} | demo prediction: {handler.predict(demo):.2f} MPa")


Model: XGBoost | demo prediction: 44.35 MPa


In [ ]:
"""
Recommendation engine — Concrete Mix Design Assistant (Phase 3)

Policy (mirrors the problem statement):
  1. COMPLIANCE FIRST — if cement < Table 5 minimum, raise cement to the
     minimum; if W/C > Table 5 maximum, cut water to the allowed maximum.
     These fixes target the violated CONSTRAINT itself (cement content or
     W/C ratio), not the strength number.
  2. STRENGTH SECOND — if the (possibly fixed) mix still predicts below the
     target grade's minimum strength, add cement: the highest-leverage
     ADJUSTABLE ingredient per the SHAP ranking (age is excluded because
     the app locks it at 28 days). The first delta estimate comes from the
     model's own local sensitivity (finite difference), then the engine
     VERIFIES by re-running the prediction and iterating in 10 kg/m3 steps
     up to a physical cap of 550 kg/m3 (just above the dataset max of 540).
  3. CONSTANT TOTAL WEIGHT — every adjustment is rebalanced through fine
     aggregate (the slack variable) so the mix still sums to the user's
     original total (~2400 kg/m3 per the PS note).
  4. HONEST LIMITS — if the target cannot be reached within the cap, the
     engine says so instead of pretending success.

Demo (run from the repo root, after Phase 1):
    python src/recommender.py
"""

from __future__ import annotations

import math
from pathlib import Path



CEMENT_CAP = 550.0        # physical cap, just above dataset max (540 kg/m3)
CEMENT_STEP = 10.0        # site-practical increment
SAFETY_MARGIN = 1.0       # MPa buffer for model uncertainty (test RMSE 4.39)
FINE_AGG_MIN = 500.0      # sanity bounds for the slack variable
FINE_AGG_MAX = 1050.0


def _rebalance(mix: dict, target_total: float, notes: list) -> None:
    """Fine aggregate is the slack variable: keep total mix weight constant."""
    others = sum(mix[k] for k in INGREDIENTS if k != "fine_aggregate")
    fine = target_total - others
    if fine < FINE_AGG_MIN:
        notes.append(
            f"Fine aggregate hit its lower bound ({FINE_AGG_MIN:.0f} kg/m3); "
            f"total mix weight rises above the original {target_total:.0f} kg/m3."
        )
        fine = FINE_AGG_MIN
    elif fine > FINE_AGG_MAX:
        notes.append(
            f"Fine aggregate hit its upper bound ({FINE_AGG_MAX:.0f} kg/m3); "
            f"total mix weight falls below the original {target_total:.0f} kg/m3."
        )
        fine = FINE_AGG_MAX
    mix["fine_aggregate"] = fine


def generate_recommendation(mix: dict, target_grade: str, model_handler) -> dict:
    """Return specific, verified ingredient adjustments for the target grade."""
    if target_grade not in IS_456_TABLE_5:
        raise ValueError(f"Unknown grade {target_grade!r}. Valid: {list(IS_456_TABLE_5)}")
    limits = IS_456_TABLE_5[target_grade]

    working = {k: float(mix[k]) for k in INGREDIENTS}
    target_total = sum(working.values())
    actions: list[str] = []
    notes: list[str] = []

    # ---- Step 1: fix violated IS 456 constraints (constraint-targeted) ----
    if working["cement"] < limits["min_cement"]:
        delta = limits["min_cement"] - working["cement"]
        working["cement"] = limits["min_cement"]
        _rebalance(working, target_total, notes)
        actions.append(
            f"Increase cement by +{delta:.1f} kg/m3 (to {limits['min_cement']:.0f}) "
            f"to meet the IS 456 minimum cement content for {target_grade}."
        )
    wc = (working["water"] / working["cement"]) if working["cement"] > 0 else float("inf")
    if wc > limits["max_wc"]:
        allowed = working["cement"] * limits["max_wc"]
        delta = working["water"] - allowed
        working["water"] = allowed
        _rebalance(working, target_total, notes)
        actions.append(
            f"Reduce water by -{delta:.1f} kg/m3 to bring W/C from {wc:.3f} down "
            f"to the IS 456 maximum of {limits['max_wc']:.2f} for {target_grade}."
        )

    pred = model_handler.predict(working)

    # ---- Step 2: fix strength shortfall (cement = primary SHAP lever) ----
    if pred < limits["min_strength"]:
        aim = limits["min_strength"] + SAFETY_MARGIN
        cement_before = working["cement"]

        # Local sensitivity of the model at this mix (includes rebalance effect)
        probe = dict(working)
        probe["cement"] = min(working["cement"] + CEMENT_STEP, CEMENT_CAP)
        _rebalance(probe, target_total, [])
        slope = 0.0
        if probe["cement"] > working["cement"]:
            slope = (model_handler.predict(probe) - pred) / (probe["cement"] - working["cement"])
        if slope > 0:
            est = min((aim - pred) / slope, CEMENT_CAP - working["cement"])
            est = math.ceil(est / CEMENT_STEP) * CEMENT_STEP  # round up to 10s
            if est > 0:
                working["cement"] += est
                _rebalance(working, target_total, notes)
                pred = model_handler.predict(working)

        # Verification loop: bounded +10 steps until target or cap
        while pred < aim and working["cement"] < CEMENT_CAP:
            working["cement"] = min(working["cement"] + CEMENT_STEP, CEMENT_CAP)
            _rebalance(working, target_total, notes)
            pred = model_handler.predict(working)

        added = working["cement"] - cement_before
        if pred >= limits["min_strength"]:
            actions.append(
                f"Increase cement by an additional +{added:.1f} kg/m3 (to "
                f"{working['cement']:.0f}) — the highest-leverage adjustable "
                f"ingredient per SHAP — to raise predicted strength to the "
                f"{target_grade} minimum of {limits['min_strength']:.0f} MPa."
            )
        else:
            actions.append(
                f"Could not reach {target_grade} even at the cement cap of "
                f"{CEMENT_CAP:.0f} kg/m3 (+{added:.1f} applied; best predicted "
                f"strength {pred:.2f} MPa). A redesign (water reduction, "
                f"admixtures, or a lower target grade) is required."
            )

    compliance = verify_is456(target_grade, pred, working["cement"], working["water"])
    return {
        "target_grade": target_grade,
        "original_mix": {k: float(mix[k]) for k in INGREDIENTS},
        "adjusted_mix": {k: round(v, 2) for k, v in working.items()},
        "actions": actions,
        "notes": notes,
        "updated_strength": round(pred, 2),
        "updated_compliance": compliance,
        "target_reached": compliance.strength_pass
        and compliance.cement_pass and compliance.wc_pass,
    }


if __name__ == "__main__":
    handler = ModelHandler()

    scenarios = [
        ("Weak mix vs M40 (compliance + strength fixes)", {
            "cement": 300.0, "slag": 0.0, "fly_ash": 0.0, "water": 190.0,
            "superplasticizer": 0.0, "coarse_aggregate": 1000.0, "fine_aggregate": 910.0,
        }, "M40"),
        ("Strong but non-compliant vs M30 (SCM-heavy, the decoupling case)", {
            "cement": 250.0, "slag": 200.0, "fly_ash": 100.0, "water": 170.0,
            "superplasticizer": 6.0, "coarse_aggregate": 1000.0, "fine_aggregate": 680.0,
        }, "M30"),
        ("Already sufficient vs M25 (no changes expected)", {
            "cement": 400.0, "slag": 100.0, "fly_ash": 0.0, "water": 160.0,
            "superplasticizer": 8.0, "coarse_aggregate": 1000.0, "fine_aggregate": 732.0,
        }, "M25"),
    ]

    for title, mix, grade in scenarios:
        before = handler.predict(mix)
        rec = generate_recommendation(mix, grade, handler)
        print(f"\n=== {title} ===")
        print(f"Initial prediction: {before:.2f} MPa | Target grade: {grade}")
        for a in rec["actions"] or ["No changes needed — mix is compliant and strong enough."]:
            print(f"  - {a}")
        for n in rec["notes"]:
            print(f"  note: {n}")
        print(f"Updated prediction: {rec['updated_strength']:.2f} MPa | "
              f"target_reached: {rec['target_reached']}")



=== Weak mix vs M40 (compliance + strength fixes) ===
Initial prediction: 26.10 MPa | Target grade: M40
  - Increase cement by +60.0 kg/m3 (to 360) to meet the IS 456 minimum cement content for M40.
  - Reduce water by -46.0 kg/m3 to bring W/C from 0.528 down to the IS 456 maximum of 0.40 for M40.
Updated prediction: 60.98 MPa | target_reached: True

=== Strong but non-compliant vs M30 (SCM-heavy, the decoupling case) ===
Initial prediction: 43.51 MPa | Target grade: M30
  - Increase cement by +70.0 kg/m3 (to 320) to meet the IS 456 minimum cement content for M30.
  - Reduce water by -26.0 kg/m3 to bring W/C from 0.531 down to the IS 456 maximum of 0.45 for M30.
Updated prediction: 59.32 MPa | target_reached: True

=== Already sufficient vs M25 (no changes expected) ===
Initial prediction: 63.08 MPa | Target grade: M25
  - No changes needed — mix is compliant and strong enough.
Updated prediction: 63.08 MPa | target_reached: True


In [4]:
%cd /content/concrete-mix-assistant
!pwd
!ls -lah

/content/concrete-mix-assistant
/content/concrete-mix-assistant
total 68K
drwxr-xr-x 8 root root 4.0K Sep  5 10:07 .
drwxr-xr-x 1 root root 4.0K Sep  5 10:07 ..
-rw-r--r-- 1 root root 8.9K Sep  5 10:07 app.py
drwxr-xr-x 2 root root 4.0K Sep  5 10:07 data
drwxr-xr-x 8 root root 4.0K Sep  5 10:07 .git
drwxr-xr-x 3 root root 4.0K Sep  5 10:07 .github
-rw-r--r-- 1 root root    1 Sep  5 10:07 .gitignore
drwxr-xr-x 2 root root 4.0K Sep  5 10:07 models
drwxr-xr-x 2 root root 4.0K Sep  5 10:07 notebooks
-rw-r--r-- 1 root root    1 Sep  5 10:07 README.md
-rw-r--r-- 1 root root  122 Sep  5 10:07 requirements.txt
drwxr-xr-x 2 root root 4.0K Sep  5 10:07 src
-rw-r--r-- 1 root root 9.7K Sep  5 10:07 train_model.py


In [5]:
!echo "----- MODEL ARTIFACTS -----"
!ls -lh models

!echo
!echo "----- INSTALLING DEPENDENCIES -----"
!pip install -r requirements.txt

----- MODEL ARTIFACTS -----
total 1.4M
-rw-r--r-- 1 root root  183 Sep  5 10:07 feature_importance.json
-rw-r--r-- 1 root root 1.9K Sep  5 10:07 metrics.json
-rw-r--r-- 1 root root 1.4M Sep  5 10:07 model.joblib
-rw-r--r-- 1 root root  275 Sep  5 10:07 shap_importance.json

----- INSTALLING DEPENDENCIES -----
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 89.1 MB/s eta 0:00:00


In [11]:
!git pull
!python -m src.recommender

Already up to date.

=== Weak mix vs M40 (compliance + strength fixes) ===
Initial prediction: 26.10 MPa | Target grade: M40
  - Increase cement by +60.0 kg/m3 (to 360.0 kg/m3) to meet the IS 456 minimum cement content for M40 (360 kg/m3).
  - Reduce water by -46.0 kg/m3 (to 144.0 kg/m3) to bring W/C from 0.528 down to the IS 456 maximum of 0.40 for M40.
  - After the IS 456 compliance adjustments, the verified predicted 28-day strength is 60.98 MPa. This meets the M40 minimum of 40 MPa, so no additional strength adjustment is required.
Updated prediction: 60.98 MPa | target_reached: True

=== Strong but non-compliant vs M30 (SCM-heavy) ===
Initial prediction: 43.51 MPa | Target grade: M30
  - Increase cement by +70.0 kg/m3 (to 320.0 kg/m3) to meet the IS 456 minimum cement content for M30 (320 kg/m3).
  - Reduce water by -26.0 kg/m3 (to 144.0 kg/m3) to bring W/C from 0.531 down to the IS 456 maximum of 0.45 for M30.
  - After the IS 456 compliance adjustments, the verified predicted 2

In [12]:
!head -30 app.py

"""
Concrete Mix Design Assistant — Streamlit application
Concreate Club ML Inductions, IIT Indore

Flow (per the problem statement):
  1. The user enters 7 mix ingredients (kg/m3) and a target IS 456 grade.
     Age is FIXED at 28 days — the app never asks for it.
  2. ML layer: XGBoost predicts the 28-day compressive strength (MPa).
  3. IS 456 layer: Table 5 checks (min cement, max W/C) displayed
     SEPARATELY from the strength prediction, so a strength failure is
     never confused with a compliance failure.
  4. If the mix falls short, the recommender proposes specific deltas and
     the app re-runs the prediction to VERIFY the fix.

Run:  streamlit run app.py
"""

import json
from pathlib import Path

import pandas as pd
import streamlit as st

from src.compliance import IS_456_TABLE_5, classify_grade, verify_is456
from src.model_handler import INGREDIENTS, ModelHandler
from src.recommender import generate_recommendation

st.set_page_config(
    page_title="Concrete Mix Desig

In [32]:
!python -m py_compile app.py

In [36]:
!pkill -f streamlit || true
!pkill -f lt || true
!fuser -k 8501/tcp || true

^C
^C


In [37]:
%cd /content/concrete-mix-assistant
!pwd

/content/concrete-mix-assistant
/content/concrete-mix-assistant


In [38]:
!python -m py_compile app.py

In [39]:
!python -c "from src.compliance import verify_is456; from src.model_handler import ModelHandler; from src.recommender import generate_recommendation; print('imports okay')"

imports okay


In [40]:
!rm -f /content/streamlit.log
!streamlit run app.py --server.address 0.0.0.0 --server.port 8501 --server.headless true --browser.gatherUsageStats false > /content/streamlit.log 2>&1 &

In [41]:
!sleep 8
!cat /content/streamlit.log

2026-09-05 10:49:54.990 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.230.14.45:8501



In [42]:
!timeout 10 curl -v http://127.0.0.1:8501

*   Trying 127.0.0.1:8501...
* Connected to 127.0.0.1 (127.0.0.1) port 8501 (#0)
> GET / HTTP/1.1
> Host: 127.0.0.1:8501
> User-Agent: curl/7.81.0
> Accept: */*
> 
* Mark bundle as not supporting multiuse
< HTTP/1.1 200 OK
< date: Sat, 05 Sep 2026 10:50:53 GMT
< server: uvicorn
< content-type: text/html; charset=utf-8
< accept-ranges: bytes
< content-length: 7459
< last-modified: Sat, 05 Sep 2026 10:09:25 GMT
< etag: "107c5340a1441a0e63955f61af7a919d"
< cache-control: no-cache
< 
<!--
 Copyright (c) Streamlit Inc. (2018-2022) Snowflake Inc. (2022-2026)

 Licensed under the Apache License, Version 2.0 (the "License");
 you may not use this file except in compliance with the License.
 You may obtain a copy of the License at

     http://www.apache.org/licenses/LICENSE-2.0

 Unless required by applicable law or agreed to in writing, software
 distributed under the License is distributed on an "AS IS" BASIS,
 WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
 See the

In [43]:
!pkill -f lt || true

^C


In [44]:
!pip install pyngrok

In [45]:
from pyngrok import ngrok

ngrok.set_auth_token("3IuEHgqgRikwGnou80IhmIm1jAb_7FPUxzhdFPSK1qY49V3V7")

In [46]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8501, "http")
print(public_url)

NgrokTunnel: "https://humble-broker-herbicide.ngrok-free.dev" -> "http://localhost:8501"
